In [0]:
CREATE OR REPLACE TABLE bronze_customers (
    customer_id INT,
    name STRING,
    age STRING,
    ingestion_time TIMESTAMP
)
USING DELTA;

In [0]:
INSERT INTO bronze_customers VALUES
(1, 'john', '25', current_timestamp()),
(2, 'sarah', '30', current_timestamp()),
(3, 'john', '25', current_timestamp()),
(4, NULL, 'invalid', current_timestamp());

In [0]:
CREATE OR REPLACE TABLE silver_customers
USING DELTA
AS
SELECT DISTINCT
    customer_id,
    INITCAP(name) AS name,
    CAST(age AS INT) AS age
FROM bronze_customers
WHERE name IS NOT NULL
  AND TRY_CAST(age AS INT) IS NOT NULL;



In [0]:
SELECT * FROM silver_customers;


-- =====================================================
-- 3. CREATE GOLD TABLE
-- Business aggregation
-- =====================================================

CREATE OR REPLACE TABLE gold_customer_summary
USING DELTA
AS
SELECT
    COUNT(*) AS total_customers,
    AVG(age) AS average_age
FROM silver_customers;


SELECT * FROM gold_customer_summary;

In [0]:
DESCRIBE HISTORY silver_customers;


-- =====================================================
-- 5. CREATE A NEW VERSION
-- =====================================================

UPDATE silver_customers
SET age = 26
WHERE customer_id = 1;


-- Check current version
SELECT * FROM silver_customers;


In [0]:
UPDATE silver_customers
SET age = 26
WHERE customer_id = 1;


-- Check current version
SELECT * FROM silver_customers;


In [0]:
DESCRIBE silver_customers

In [0]:
SELECT * FROM silver_customers WHERE age= 24

In [0]:
DESCRIBE silver_customers

In [0]:
DESCRIBE HISTORY silver_customers;

In [0]:
OPTIMIZE silver_customers;


In [0]:
%python
# Unity Catalog Volume path format: /Volumes/<catalog>/<schema>/<volume>/<file_path>
# First, create a volume in practice_catalog.default:
# CREATE VOLUME IF NOT EXISTS practice_catalog.default.superstore_data;

# Then upload your file to that volume and use:
file_path = "/Volumes/practice_catalog/default/supestoredata"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

display(df)

Replacing colun names with space to underscore

In [0]:
%python
df = df.toDF(*[
    col.strip()
       .lower()
       .replace(" ", "_")
       .replace("-", "_")
    for col in df.columns
])

display(df)

In [0]:
%python
df.write.format("delta").mode("overwrite").saveAsTable("superstore_customers_bronze")

In [0]:
UPDATE superstore_customers_bronze

SET profit = 999.99
WHERE row_id = 5004;

In [0]:
DESCRIBE HISTORY superstore_customers_bronze

In [0]:
SELECT row_id, profit
FROM workspace.default.superstore_customers_bronze VERSION AS OF 0
WHERE row_id = 5004;

In [0]:
%python
from pyspark.sql import Row

new_data = [
    Row(
        row_id=99999,
        order_id="TEST-001",
        order_date="2026-08-29",
        new_column="TEST"
    )
]

test_df = spark.createDataFrame(new_data)

display(test_df)

In [0]:
%python
test_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.superstore_customers_bronze")

In [0]:
%python
test_df.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("workspace.default.superstore_customers_bronze")

In [0]:
%python
spark.table("workspace.default.superstore_customers_bronze").printSchema()

In [0]:
OPTIMIZE workspace.default.superstore_customers_bronze

In [0]:
DESCRIBE HISTORY workspace.default.superstore_customers_bronze;

In [0]:
CREATE OR REPLACE TABLE workspace.default.superstore_customers_silver AS

SELECT DISTINCT
    row_id,
    order_id,
    TO_DATE(order_date, 'MM-dd-yyyy') AS order_date,
    TO_DATE(ship_date, 'MM/dd/yyyy') AS ship_date,
    ship_mode,
    customer_id,
    customer_name,
    segment,
    country,
    city,
    state,
    postal_code,
    region,
    product_id,
    category,
    sub_category,
    product_name,
    TRY_CAST(sales AS DOUBLE) AS sales,
    TRY_CAST(quantity AS INT) AS quantity,
    TRY_CAST(discount AS DOUBLE) AS discount,
    TRY_CAST(profit AS DOUBLE) AS profit

FROM workspace.default.superstore_customers_bronze

WHERE order_id IS NOT NULL
  AND TRY_CAST(sales AS DOUBLE) >= 0;

In [0]:
select  count(*) from superstore_customers_silver

In [0]:
CREATE OR REPLACE TABLE workspace.default.gold_sales_summary AS

SELECT
    category,
    region,
    COUNT(*) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    AVG(discount) AS average_discount
FROM workspace.default.superstore_customers_silver
GROUP BY category, region;

In [0]:
select * from gold_sales_summary

In [0]:
%python
print("CI/CD practices")